In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    r2_score
)

df = pd.read_csv(r"C:\Users\lenovo\OneDrive\Desktop\mlops_day1\data\data.csv")
    


df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [7]:
# Drop 'Unnamed: 0' (or the exact column name in your DataFrame)
df = df.drop(columns=['Unnamed: 0'])

In [11]:


# Feature selection and target definition

X = df[["TV", "radio", "newspaper"]]
y = df["sales"]


# Train-test split
xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=56
)

# Configure MLflow tracking URI (local SQLite database)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Set or create an experiment name
mlflow.set_experiment("Advertising Sales Prediction")


2026/09/13 11:09:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/13 11:09:22 INFO mlflow.store.db.utils: Updating database tables
2026/09/13 11:09:27 INFO mlflow.tracking.fluent: Experiment with name 'Advertising Sales Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/lenovo/OneDrive/Desktop/mlops_day1/notebooks/mlruns/1', creation_time=1789277967924, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789277967924, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
import mlflow

with mlflow.start_run(run_name="Linear Regression"):

    model = LinearRegression()
    model.fit(xtrain, ytrain)

    y_pred = model.predict(xtest)

    # Note: Use root_mean_squared_error(ytest, y_pred) if on scikit-learn >= 1.4,
    # or root_mean_squared_error = mean_squared_error(ytest, y_pred, squared=False) for older versions.
    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

In [13]:
with mlflow.start_run(run_name="Ridge Regression"):

    model = Ridge(alpha=1.0)
    model.fit(xtrain, ytrain)

    ypred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, ypred)
    r2 = r2_score(ytest, ypred)

    # Log the parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Log the metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    # Log Artifacts
    mlflow.sklearn.log_model(sk_model=model, name="Ridge_Reg_Model")

Artifacts are the concrete output files generated by a run.

Now, we use the autologging feature in MLflow.

Autologging allows MLflow to automatically capture much of the information.

In [14]:
# Turn on Scikit-learn autologging.
mlflow.sklearn.autolog()

In [15]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:

    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    model.fit(xtrain, ytrain)

    test_pred = model.predict(xtest)

    test_rmse = root_mean_squared_error(ytest, test_pred)
    test_mae = mean_absolute_error(ytest, test_pred)
    test_r2 = r2_score(ytest, test_pred)

    # Custom project metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
        }
    )

2026/09/13 11:43:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registering the Model

The MLflow Model Registry is a central hub (like an app store or catalog) to keep all production-ready models in one shared, searchable place instead of scattered across folders or runs.

Among all the experiments you perform, register the final selected model.

In a production environment, the models are continuously trained. This means the registered models would have many versions:
Advertising_Sales_Model



 ├── Version 1
 ├── Version 2
 ├── Version 3
 └── Version 4

In [16]:
# For registering the model, we require the model URI.
# URI is a unique identifier for a model.

run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)


# Now let's register the model

registered_model = mlflow.register_model(
    model_uri=model_uri, name="Advertising_Sales_Model"
)


Successfully registered model 'Advertising_Sales_Model'.
2026/09/13 11:55:59 WARNING mlflow.tracking._model_registry.fluent: Run with id f624601a2d9c4625beb207322930dd51 has no artifacts at artifact path 'model', registering model based on models:/m-e65bfb51eb7f4301bbf6813a47c633e9 instead


runs:/f624601a2d9c4625beb207322930dd51/model


Created version '1' of model 'Advertising_Sales_Model'.


Model Aliases

Model Aliasing gives a nickname (like "current_best" or "production") to a specific version of the registered model.

Instead of typing exact numbers like Version 1, Version 2, or Version 15, you just use the nickname.
Advertising_Sales_Model
 ├── Version 1
 ├── Version 2
 ├── Version 3  ← champion
 └── Version 4  ← challenger

 champion means:

the currently preferred model.

challenger means:

a new candidate being evaluated as a possible replacement.

In [17]:
from mlflow import MlflowClient

client = MlflowClient()

# 1. Assign an alias to a specific version
# (Sets the alias 'champion' to Version 1 of 'Advertising_Sales_Model')
client.set_registered_model_alias(
    name="Advertising_Sales_Model",
    alias="champion",
    version="1"
)